# WhatsApp Agent

Steps

1. Login / Register: https://www.twilio.com/en-us
2. `Messaging` -> `Try it out` -> `Send a WhatsApp message`

In [6]:
from langchain_openai import ChatOpenAI

from fastapi import FastAPI, Request
from fastapi.responses import Response
from twilio.twiml.messaging_response import MessagingResponse

from dotenv import load_dotenv
import os
load_dotenv(override=True)

True

In [7]:
app = FastAPI()
# Initialize LLMs (Using Groq API as configured previously)
def get_groq_llm(model_name="openai/gpt-oss-20b"):
    return ChatOpenAI(
        model=model_name,
        base_url="https://api.groq.com/openai/v1",
        api_key=os.getenv("GROQ_API_KEY"),
        temperature=0.7,
        max_tokens=2000
    )


extraction_llm = get_groq_llm("openai/gpt-oss-20b")
chat_llm = get_groq_llm("meta-llama/llama-4-scout-17b-16e-instruct")

In [8]:
@app.post("/webhook/whatsapp")
async def webhook_whatsapp(request: Request):
    form_data = await request.form()

    user_msg = form_data.get("Body")
    from_num = form_data.get("From")
    print(f"Received WhatsApp message from {from_num}: {user_msg}")

    if not user_msg:
        reply = "I didn't receive your message. Please try again."
    else:
        # Use the chat LLM to generate a response
        response = chat_llm.invoke(user_msg)
        reply = response.content

    # Create a TwiML response
    twilio_resp = MessagingResponse()
    twilio_resp.message(reply)

    return Response(
        content=str(twilio_resp),
        media_type="application/xml"
    )

In [2]:
import os
from dotenv import load_dotenv
from twilio.rest import Client

load_dotenv(override=True)

account_sid = os.getenv('TWILIO_ACCOUNT_SID')
auth_token = os.getenv('TWILIO_AUTH_TOKEN')

# --- DEBUG CHECK ---
if not account_sid or not auth_token:
    print("❌ ERROR: Credentials are None. Python cannot find your .env file or variables.")
else:
    print(f"✅ Credentials found! SID starts with: {account_sid[:5]}...")

client = Client(account_sid, auth_token)

✅ Credentials found! SID starts with: ACf0e...


In [5]:
from twilio.rest import Client

account_sid = os.getenv('TWILIO_ACCOUNT_SID')
auth_token = os.getenv('TWILIO_AUTH_TOKEN')
client = Client(account_sid, auth_token)

message = client.messages.create(
  from_='whatsapp:+14155238886',
  content_sid='HXb5b62575e6e4ff6129ad7c8efe1f983e',
  content_variables='{"1":"12/1","2":"3pm"}',
  to='whatsapp:+917337889111'
)

print(message.sid)

TwilioRestException: HTTP 401 error: Unable to create record: Authenticate